In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/dataset/"

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import cv2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

In [ ]:
# -----------------------------
# Constants
# -----------------------------
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 4

In [ ]:
# -----------------------------
# Load metadata (ONE ROW PER CLASS)
# -----------------------------
df = pd.read_csv(path+'metadata.csv')

label_map = {
    "non_dem": "NonDemented",
    "very_mild_dem": "VeryMildDemented",
    "mild_dem": "MildDemented",
    "moderat_dem": "ModerateDemented"
}

df["folder_label"] = df["label"].map(label_map)

# Normalize age
df["age"] = df["age"] / 100.0

# Build metadata lookup: label -> metadata
label_to_metadata = {
    row.folder_label: np.array(
        [row.age, row.sex, row.family_history, row.apoe],
        dtype=np.float32
    )
    for _, row in df.iterrows()
}


In [ ]:
# -----------------------------
# Build dataset arrays
# -----------------------------
image_paths = []
labels = []
metadata = []

class_map = {
    "NonDemented": 0,
    "VeryMildDemented": 1,
    "MildDemented": 2,
    "ModerateDemented": 3
}

for folder_label, class_idx in class_map.items():
    folder = Path(path+f"train/{folder_label}")
    meta = label_to_metadata[folder_label]

    for img_path in folder.iterdir():
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                image_paths.append(str(img_path))
                labels.append(class_idx)
                metadata.append(meta)

# Convert metadata once (safe, small)
metadata = np.array(metadata, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

print("Total samples:", len(image_paths))
print("Metadata shape:", metadata.shape)

Total samples: 37053
Metadata shape: (37053, 4)


In [ ]:
# -----------------------------
# Image loading + preprocessing
# -----------------------------
def load_and_preprocess(path, label, meta):
    # Load image
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    # One-hot encode label
    label = tf.one_hot(label, NUM_CLASSES)

    # Return ((image, metadata), label)
    return (img, meta), label

In [ ]:
# -----------------------------
# Build tf.data.Dataset
# -----------------------------
dataset = tf.data.Dataset.from_tensor_slices(
    (image_paths, labels, metadata)
)

dataset = dataset.map(
    load_and_preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

dataset = dataset.shuffle(buffer_size=1000)
dataset = dataset.batch(BATCH_SIZE)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
# -----------------------------
# Inspect one batch (sanity check)
# -----------------------------
for (img_batch, meta_batch), label_batch in dataset.take(1):
    print("Image batch:", img_batch.shape)
    print("Metadata batch:", meta_batch.shape)
    print("Label batch:", label_batch.shape)

In [ ]:
# -----------------------------
# Class imbalance correction
# -----------------------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)

Class weights: {0: np.float64(0.72369140625), 1: np.float64(0.8270758928571429), 2: np.float64(3.03414674091058), 3: np.float64(0.926325)}


In [ ]:
# -----------------------------
# Model: CNN branch
# -----------------------------
image_input = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

cnn_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_tensor=image_input
)
cnn_base.trainable = False

x = GlobalAveragePooling2D()(cnn_base.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)

In [ ]:
# -----------------------------
# Model: Metadata branch
# -----------------------------
meta_input = Input(shape=(4,))
y = Dense(32, activation="relu")(meta_input)
y = Dense(16, activation="relu")(y)

In [ ]:
# -----------------------------
# Fusion
# -----------------------------
combined = Concatenate()([x, y])
z = Dense(128, activation="relu")(combined)
z = Dropout(0.5)(z)
output = Dense(4, activation="softmax")(z)

model = Model(inputs=[image_input, meta_input], outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 128, 128,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 128, 128,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 128, 128,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 129, 129,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 64,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 64,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 64, 64,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 64, 64,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 64, 64,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 64, 64,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 64, 64,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 64, 64,    │        512 │ block1a_se_excit

 Total params: 4,413,655 (16.84 MB)

 Trainable params: 364,084 (1.39 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# -----------------------------
# Train
# -----------------------------
model.fit(
    dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
)

Epoch 1/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 3032s 1s/step - accuracy: 0.9525 - loss: 0.1723
Epoch 2/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 766s 329ms/step - accuracy: 0.8319 - loss: 0.5149
Epoch 3/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 800s 328ms/step - accuracy: 0.8664 - loss: 0.4156
Epoch 4/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 757s 325ms/step - accuracy: 0.8955 - loss: 0.4236
Epoch 5/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 780s 335ms/step - accuracy: 0.8903 - loss: 0.3845
Epoch 6/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 787s 338ms/step - accuracy: 0.9003 - loss: 0.3390
Epoch 7/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 800s 344ms/step - accuracy: 0.9089 - loss: 0.2837
Epoch 8/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 778s 335ms/step - accuracy: 0.9283 - loss: 0.2233
Epoch 9/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 774s 333ms/step - accuracy: 0.9352 - loss: 0.2095
Epoch 10/10
2316/2316 ━━━━━━━━━━━━━━━━━━━━ 804s 345ms/step - accuracy: 0.9393 - loss: 0.1997


In [ ]:
#model.save("multimodal_alzheimer_model.h5")
model.save('multimodal_alzheimer_model.keras')
#print("Model saved: multimodal_alzheimer_model.h5")

Model saved: multimodal_alzheimer_model.h5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
